# CTD Data Grapher

1. Open the Table of Contents
2. **Click the play/run button next to 'CTD Data Grapher'** - this will install required packages for Google Colab to plot and graph data.
3. **Click the play/run button next to 'CTD Setup & Site Info'** - this will allow you to select your CTD type and enter text into the required fields.
4. Select your CTD setup — **UW-Tacoma** or **UW-Friday Harbor Labs**.
5. Type in the name of the location surveyed - this name will appear on the graphs.
6. Enter the name of the excel file - Do not include '.xlsx'
   * This file needs to be in .xlsx format.
   * This data should have been imported to Excel from a .cnv file.
   * Each sheet in this .xlsx file should be untouched, imported CTD data.
   * **The names used in the legends of the graphs come from the labels of the sheets.**
   * Give sheets the names of the stations.
7. Enter name of PDF file to be saved.
   * Do not include '.pdf'

In [1]:
pip install ipywidgets openpyxl #these are required packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.4 MB/s eta 0:00:00


In [2]:
import ipywidgets as widgets
import pandas as pd

In [3]:
# CTD setup selector
ctd_setup = widgets.RadioButtons(
    options=['UW-Tacoma', 'UW-Friday Harbor'],
    description='CTD Setup:',
    disabled=False
)

# User input fields
location = widgets.Text(
    value='',
    placeholder='e.g. Quartermaster Harbor',
    description='Location:',
    disabled=False
)
fileName = widgets.Text(
    value='',
    placeholder='e.g. CTD_Data_QMH',
    description='Excel File:',
    disabled=False
)
file_CTD_PDF = widgets.Text(
    value='',
    placeholder='e.g. CTD_Graphs_QMH',
    description='PDF File:',
    disabled=False
)

# CTD Setup & Site Info
#### Select your CTD setup, then enter the location name, Excel file name, and desired PDF file name.
* Do not include '.xlsx' or '.pdf'

In [5]:
display(ctd_setup)
display(location)
display(fileName)
display(file_CTD_PDF)

RadioButtons(description='CTD Setup:', options=('UW-Tacoma', 'UW-Friday Harbor'), value='UW-Tacoma')

Text(value='', description='Location:', placeholder='e.g. Quartermaster Harbor')

Text(value='', description='Excel File:', placeholder='e.g. CTD_Data_QMH')

Text(value='', description='PDF File:', placeholder='e.g. CTD_Graphs_QMH')

# Google Drive Access
## Google will ask for permission to see your file in Google Drive.
#### The file name must match that which is entered.

* The file holding the CTD data sheets will be accessed straight from your Google Drive
* The PDF file with the plotted CTD data will be saved to your Google Drive.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Column mappings and data start row for each CTD setup
configs = {
    'UW-Tacoma': {
        'data_start_row': 559,
        'col': {
            'depth': 14,  # O
            'temp':  2,   # C
            'sal':   10,  # K
            'dens':  12,  # M
            'do':    7,   # H
            'fluor': 5,   # F
            'trans': 6    # G
        }
    },
    'UW-Friday Harbor': {
        'data_start_row': 219,
        'col': {
            'depth': 15,  # P
            'temp':  2,   # C
            'sal':   11,  # L
            'dens':  13,  # N
            'do':    8,   # I
            'fluor': 5,   # F
            'turb':  6,   # G
            'pH':    7    # H
        }
    }
}

setup = ctd_setup.value
cfg = configs[setup]
col = cfg['col']
data_start = cfg['data_start_row']

print(f"Using config: {setup}")

# Load the Excel file
xl = pd.ExcelFile('drive/My Drive/' + fileName.value + '.xlsx')
sheets = xl.sheet_names

# Initialize data dictionaries
depth_data = {}
temp_data  = {}
sal_data   = {}
dens_data  = {}
do_data    = {}
fluor_data = {}

# Setup-specific variables
if setup == 'UW-Tacoma':
    trans_data = {}
else:
    turb_data = {}
    pH_data   = {}

# Parse each station sheet
for sheet in sheets:
    df   = xl.parse(sheet, header=None)
    rows = df.iloc[data_start:, :]

    depth_data[sheet] = rows.iloc[:, col['depth']].reset_index(drop=True)
    temp_data[sheet]  = rows.iloc[:, col['temp']].reset_index(drop=True)
    sal_data[sheet]   = rows.iloc[:, col['sal']].reset_index(drop=True)
    dens_data[sheet]  = rows.iloc[:, col['dens']].reset_index(drop=True)
    do_data[sheet]    = rows.iloc[:, col['do']].reset_index(drop=True)
    fluor_data[sheet] = rows.iloc[:, col['fluor']].reset_index(drop=True)

    if setup == 'UW-Tacoma':
        trans_data[sheet] = rows.iloc[:, col['trans']].reset_index(drop=True)
    else:
        turb_data[sheet] = rows.iloc[:, col['turb']].reset_index(drop=True)
        pH_data[sheet]   = rows.iloc[:, col['pH']].reset_index(drop=True)

# Transfer to DataFrames
depth_df = pd.DataFrame(depth_data)
temp_df  = pd.DataFrame(temp_data)
sal_df   = pd.DataFrame(sal_data)
dens_df  = pd.DataFrame(dens_data)
do_df    = pd.DataFrame(do_data)
fluor_df = pd.DataFrame(fluor_data)

if setup == 'UW-Tacoma':
    trans_df = pd.DataFrame(trans_data)
else:
    turb_df = pd.DataFrame(turb_data)
    pH_df   = pd.DataFrame(pH_data)

Using config: UW-Friday Harbor


FileNotFoundError: [Errno 2] No such file or directory: 'drive/My Drive/CTD_Data_FHL.xlsx'

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

# Build dataset dict based on selected setup
datasets = {
    'Temperature (°C)':      temp_df,
    'Salinity (PSU)':        sal_df,
    'Density (kg/m³)':       dens_df,
    'Dissolved Oxygen (mL/L)': do_df,
    'Fluorescence (mg/m³)':  fluor_df,
}

if setup == 'UW-Tacoma':
    datasets['Transmissivity (%)'] = trans_df
else:
    datasets['Turbidity (NTU)'] = turb_df
    datasets['pH']              = pH_df

with PdfPages('/content/drive/My Drive/' + file_CTD_PDF.value + '.pdf') as pdf:
    for title, df in datasets.items():
        fig, ax = plt.subplots(figsize=(8, 10))

        for station in sheets:
            x = pd.to_numeric(df[station], errors='coerce')
            y = pd.to_numeric(depth_df[station], errors='coerce')

            mask = x.notna() & y.notna()
            x = x[mask].values
            y = y[mask].values

            sort_idx = np.argsort(y)
            x, y = x[sort_idx], y[sort_idx]

            spline   = make_interp_spline(y, x, k=3)
            y_smooth = np.linspace(y.min(), y.max(), 300)
            x_smooth = spline(y_smooth)
            ax.plot(x_smooth, y_smooth, label=station, linewidth=1.5)

        ax.invert_yaxis()
        ax.set_xlabel(title)
        ax.xaxis.set_label_position('top')
        ax.xaxis.tick_top()
        ax.set_ylabel('Depth (m)')
        ax.set_title(location.value + ": " +
                     (title[:title.index('(')] if '(' in title else title) +
                     "Profiles")
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"PDF saved to Google Drive: {file_CTD_PDF.value}.pdf")